# 🏍️ MARS-AI: Motorcycle Viewpoint Classification — Full Training Pipeline

**Author:** Kao Panboonyuen  
**Framework:** TensorFlow / Keras  
**Runtime target:** Google Colab (GPU) — full notebook trains in **~5–15   minutes**.   
**Dataset:** [`marsail_motorcycle_dataset.zip`](https://github.com/kaopanboonyuen/MARS-LEARN/blob/main/dataset/marsail_motorcycle_dataset.zip) — 4 classes: `back`, `front`, `left`, `right`.

---

### 🎯 What this notebook teaches

This is a **teaching notebook** built to walk a team through an end-to-end, best-practice image
classification pipeline — from raw zip file to a deployed, explainable model. It intentionally
uses **small epoch counts + small image sizes** so it finishes fast in a live class session. Every
place where you'd scale things up for a *real* production model is marked with a
> 🔧 **PRODUCTION TIP** callout — telling you exactly what to change and why it makes the model
more robust and generalizable if you let it train for hours/days instead of minutes.

By the end, you will have:
- Trained **5 models** and compared them head-to-head
- Learned **why** and **how** to write a custom loss function (Focal Loss), with the math
- Learned how to read **Grad-CAM** heatmaps to debug *why* a model is wrong
- Run a **real-world mock inference** on an image pulled straight from Wikipedia
- A cheat-sheet of tricks for making this production-grade

---

## 📑 Table of Contents

1. [Introduction & Objectives](#1)
2. [Environment Setup & Imports](#2)
3. [Dataset Download & Preparation](#3)
4. [Exploratory Data Analysis (EDA)](#4)
5. [Preprocessing, Normalization & Data Splits](#5)
6. [Data Augmentation Pipeline](#6)
7. [🧠 The Model Zoo — 5 Architectures](#7)
   - 7.1 Model 1 — ResNet50 (Transfer Learning)
   - 7.2 Model 2 — DenseNet121 (Transfer Learning)
   - 7.3 Model 3 — Vision Transformer (Fast/Tiny, trained from scratch)
   - 7.4 Model 4 — `MARS_AI_Architecture` (Our Custom CNN)
   - 7.5 Model 5 — ResNet50 + **Custom Focal Loss** (Class-Imbalance Specialist)
8. [📐 Deep Dive: Custom Loss Functions & the Math of Focal Loss](#8)
9. [Training Loop — All 5 Models](#9)
10. [📊 Model Comparison — Accuracy, Precision, Recall, F1, AUC/ROC](#10)
11. [🔍 Error Analysis — Confusion Matrix & Misclassified Samples](#11)
12. [🔥 Grad-CAM — Visual Explainability](#12)
13. [🌍 Mock Real-World Inference (image pulled from the web)](#13)
14. [🎓 Key Takeaways & Best-Practice Cheat Sheet](#14)
15. [🚀 Appendix: Production-Robustness Tricks (negative classes, long-training schedule, TTA, etc.)](#15)

---


<a id="1"></a>
## 1. Introduction & Objectives

**Task:** Classify a photo of a motorcycle into the viewpoint it was shot from — `back`, `front`,
`left`, or `right`. This is a **4-class single-label image classification** problem.

**Why it matters (real use case):** viewpoint classification is a common pre-processing step in
automotive/motorcycle inspection pipelines (insurance claim photo validation, marketplace listing
QC, damage-detection pipelines that need a canonical view first).

**Best-practice pipeline we'll follow (industry standard):**

| Stage | What we do | Why |
|---|---|---|
| EDA | Count images/class, inspect sizes, sample images | Catch imbalance & bad data early |
| Split | Stratified train/val/test | Prevent leakage, fair evaluation |
| Preprocessing | Resize, rescale to model's expected input | Each backbone expects a specific input scale |
| Augmentation | Flip/rotate/zoom/brightness | Reduce overfitting on a small dataset |
| Modeling | Compare 5 architectures | No single model wins every task — always benchmark |
| Custom loss | Focal Loss for imbalance | Cross-entropy is a poor choice when classes are skewed |
| Evaluation | Acc/Precision/Recall/F1/AUC/ROC | Accuracy alone is misleading under imbalance |
| Error analysis | Confusion matrix + visualize wrong images | Understand *why*, not just *how much*, it fails |
| Explainability | Grad-CAM | Make sure the model looks at the motorcycle, not the background |
| Inference demo | Real image from the internet | Sanity-check the model "in the wild" |

> 🔧 **PRODUCTION TIP:** Everything in this notebook is deliberately shrunk down to fit a **live
> class demo (5–15 min total runtime)**. Look out for the 🔧 callouts throughout — each tells you
> the exact knob to turn (epochs, image size, unfreezing layers, dataset size) to turn this into a
> production-grade, robust, well-generalizing model if you have hours/days of training budget.


<a id="2"></a>
## 2. Environment Setup & Imports

> ⚡ **Colab tip:** `Runtime → Change runtime type → T4 GPU` before running. Everything below is
> written to also work on CPU (just slower), so the notebook won't crash if you forget.


In [ ]:
# Colab: make sure GPU is available (optional but recommended)
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


In [ ]:
# ---- Core imports ----
import os, io, re, zipfile, glob, math, time, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

import requests
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import # INSERT_YOUR_MODEL_HERE
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, precision_score,
    recall_score, accuracy_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Global config (kept small on purpose for a fast live demo) ----
IMG_SIZE       = # INSERT YOUR LUCKY NUMBER               # 🔧 PRODUCTION TIP: bump to 224 for transfer-learning backbones
BATCH_SIZE     = # INSERT YOUR LUCKY NUMBER 
EPOCHS_DEMO    = # INSERT YOUR LUCKY NUMBER            # 🔧 PRODUCTION TIP: 30-100+ with EarlyStopping for real training
CLASS_NAMES    = ["back", "front", "left", "right"]
NUM_CLASSES    = len(CLASS_NAMES)
AUTOTUNE       = tf.data.AUTOTUNE

print("Config -> IMG_SIZE:", IMG_SIZE, "| BATCH_SIZE:", BATCH_SIZE, "| EPOCHS_DEMO:", EPOCHS_DEMO)


<a id="3"></a>
## 3. Dataset Download & Preparation

We pull the zip directly from the MARS-LEARN GitHub repo and unzip it into `/content/data`
(on Colab) or `./data` locally.


In [ ]:
DATASET_NAME = "YOUR_DATASET"
DATA_URL = f"https://github.com/kaopanboonyuen/MARS-LEARN/raw/main/dataset/{DATASET_NAME}.zip"

DATA_DIR = "./data"
ZIP_PATH = "./marsail_motorcycle_dataset.zip"

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading dataset...")
    r = requests.get(DATA_URL, timeout=120)
    r.raise_for_status()
    with open(ZIP_PATH, "wb") as f:
        f.write(r.content)
    print(f"Downloaded {len(r.content)/1e6:.2f} MB")
else:
    print("Zip already present, skipping download.")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(DATA_DIR)

print("Extracted. Top-level contents of", DATA_DIR, ":", os.listdir(DATA_DIR))


In [ ]:
# Auto-locate the actual folder that contains back/front/left/right sub-dirs
# (handles the zip possibly extracting into a nested folder)
def find_dataset_root(base_dir, expected_classes=CLASS_NAMES):
    for root, dirs, files in os.walk(base_dir):
        if all(c in dirs for c in expected_classes):
            return root
    raise FileNotFoundError("Could not find a folder containing back/front/left/right sub-dirs.")

DATASET_ROOT = find_dataset_root(DATA_DIR)
print("Dataset root:", DATASET_ROOT)
for c in CLASS_NAMES:
    n = len(glob.glob(os.path.join(DATASET_ROOT, c, "*")))
    print(f"  {c:6s} : {n} images")


<a id="4"></a>
## 4. Exploratory Data Analysis (EDA)

Before writing a single line of model code, we **always** look at the data first:
1. Class balance (do we have an imbalance problem? — spoiler: yes, this is *why* we build Model 5)
2. Image dimensions / aspect ratios (do we need to worry about weird resolutions?)
3. Actual sample images per class (sanity check labels are correct)
4. Corrupt-file check (broken images will crash training halfway through if you don't catch this early)


In [ ]:
# ---- 4.1 Class balance ----
counts = {c: len(glob.glob(os.path.join(DATASET_ROOT, c, "*"))) for c in CLASS_NAMES}
df_counts = pd.DataFrame(list(counts.items()), columns=["class", "count"]).sort_values("count", ascending=False)
display(df_counts)

plt.figure(figsize=(6,4))
sns.barplot(data=df_counts, x="class", y="count", palette="viridis")
plt.title("Class distribution")
plt.ylabel("# images")
for i, v in enumerate(df_counts["count"]):
    plt.text(i, v + 0.5, str(v), ha="center")
plt.tight_layout()
plt.show()

imbalance_ratio = df_counts["count"].max() / df_counts["count"].min()
print(f"Imbalance ratio (max/min class count): {imbalance_ratio:.2f}x")
if imbalance_ratio > #INSERT_YOUR_SCORE:
    print("=> Meaningful class imbalance detected. This is exactly why Model 5 uses Focal Loss (Section 8).")


In [ ]:
# ---- 4.2 Image dimension / aspect-ratio check ----
sample_paths = []
for c in CLASS_NAMES:
    sample_paths += glob.glob(os.path.join(DATASET_ROOT, c, "*"))[:30]  # sample for speed

dims = []
for p in sample_paths:
    try:
        with Image.open(p) as im:
            dims.append(im.size)  # (w, h)
    except Exception as e:
        print("Corrupt/unreadable:", p, e)

dims = np.array(dims)
print("Sampled", len(dims), "images")
print("Width  -> min:", dims[:,0].min(), "max:", dims[:,0].max(), "mean:", dims[:,0].mean().round(1))
print("Height -> min:", dims[:,1].min(), "max:", dims[:,1].max(), "mean:", dims[:,1].mean().round(1))


In [ ]:
# ---- 4.3 Corrupt-file scan across the FULL dataset (important before training!) ----
bad_files = []
for c in CLASS_NAMES:
    for p in glob.glob(os.path.join(DATASET_ROOT, c, "*")):
        try:
            with Image.open(p) as im:
                im.verify()
        except Exception:
            bad_files.append(p)

print(f"Found {len(bad_files)} corrupt/unreadable file(s).")
for p in bad_files:
    print("Removing corrupt file:", p)
    os.remove(p)


In [ ]:
# ---- 4.4 Visual sanity check: sample grid per class ----
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, 3*NUM_CLASSES))
for row, c in enumerate(CLASS_NAMES):
    paths = glob.glob(os.path.join(DATASET_ROOT, c, "*"))[:5]
    for col in range(5):
        ax = axes[row, col]
        if col < len(paths):
            img = mpimg.imread(paths[col])
            ax.imshow(img)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(c, fontsize=12)
    axes[row, 0].set_title(c, loc="left", fontsize=13, fontweight="bold")
plt.suptitle("Sample images per class", fontsize=15)
plt.tight_layout()
plt.show()


> 🔧 **PRODUCTION TIP:** In a real project, EDA also includes duplicate-image detection
> (perceptual hashing, e.g. `imagehash`), EXIF-orientation checks, and looking for
> near-duplicate frames from video bursts (which cause train/test leakage if split randomly).


<a id="5"></a>
## 5. Preprocessing, Normalization & Data Splits

**Normalization matters a lot** and depends on the backbone:

| Model | Expected input | How we normalize |
|---|---|---|
| ResNet50 | Caffe-style, BGR, mean-subtracted | `tf.keras.applications.resnet50.preprocess_input` |
| DenseNet121 | `[0,1]` then ImageNet mean/std | `tf.keras.applications.densenet.preprocess_input` |
| Vision Transformer (ours, from scratch) | `[0,1]` | simple `Rescaling(1/255)` |
| MARS_AI_Architecture (ours, from scratch) | `[0,1]` | simple `Rescaling(1/255)` |
| ResNet50 + Focal Loss | same as ResNet50 | `resnet_preprocess` |

We build **one shared stratified split** (train/val/test) so every model is evaluated on
exactly the same held-out images — a must for a fair comparison.


In [ ]:
# ---- 5.1 Build a DataFrame of (filepath, label) ----
records = []
for c in CLASS_NAMES:
    for p in glob.glob(os.path.join(DATASET_ROOT, c, "*")):
        records.append({"filepath": p, "label": c})
df = pd.DataFrame(records)
print("Total images:", len(df))
df.head()


In [ ]:
# ---- 5.2 Stratified train / val / test split (70 / 15 / 15) ----
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, dict(d["label"].value_counts()))


In [ ]:
# ---- 5.3 Encode labels ----
label_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
idx_to_label = {i: c for c, i in label_to_idx.items()}

train_df["label_idx"] = train_df["label"].map(label_to_idx)
val_df["label_idx"]   = val_df["label"].map(label_to_idx)
test_df["label_idx"]  = test_df["label"].map(label_to_idx)

# ---- 5.4 Class weights (used later to help with imbalance for the "normal" models) ----
from sklearn.utils.class_weight import compute_class_weight
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=#YOUR_LABEL_CLASS
)
CLASS_WEIGHTS = {i: w for i, w in enumerate(class_weights_arr)}
print("Class weights (balanced):", CLASS_WEIGHTS)


In [ ]:
# ---- 5.5 tf.data pipeline builder ----
def make_dataset(df, img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=False,
                  augment=False, preprocess_fn=None):
    paths = df["filepath"].values
    labels = df["label_idx"].values

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        return img, label

    def _augment(img, label):
        img = tf.image.random_flip_left_right(img) if False else img  # NOTE: see caution below
        img = tf.image.random_brightness(img, max_delta=0.15)
        img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
        img = tf.clip_by_value(img, 0.0, 255.0)
        return img, label

    def _preprocess(img, label):
        if preprocess_fn is not None:
            img = preprocess_fn(img)
        else:
            img = img / 255.0
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
    ds = ds.map(_load, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(_preprocess, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


> ⚠️ **Important domain-specific caution:** we deliberately do **NOT** apply a random
> left-right flip here, even though it's a "standard" augmentation elsewhere. Why? Because
> flipping a `left`-view motorcycle horizontally turns it into a `right`-view motorcycle
> (and vice-versa) — that would silently **corrupt the labels**! Always think about whether an
> augmentation is *label-preserving* for your specific task before using it.


<a id="6"></a>
## 6. Data Augmentation Pipeline

Label-safe augmentations for this task: small rotations, zoom, brightness/contrast jitter,
and slight translation. These simulate camera-angle and lighting variance **without** flipping
left/right semantics.


In [ ]:
# Keras preprocessing layers version (used inside model graphs so it also augments on GPU)
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.05),          # ~+-18 degrees max, mild
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name="data_augmentation")

# Quick visual check of augmentation on one sample image
sample_path = train_df.iloc[0]["filepath"]
raw = tf.io.decode_jpeg(tf.io.read_file(sample_path), channels=3)
raw = tf.image.resize(raw, [IMG_SIZE, IMG_SIZE])

plt.figure(figsize=(12, 3))
for i in range(6):
    aug = data_augmentation(tf.expand_dims(raw, 0), training=True)[0]
    plt.subplot(1, 6, i+1)
    plt.imshow(tf.clip_by_value(aug, 0, 255).numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle(f"Augmentation preview -- label: {train_df.iloc[0]['label']}")
plt.tight_layout()
plt.show()


> PRODUCTION TIP: for a long-training-budget run, also add
> `RandomCutout`/`RandomErasing`, `MixUp`/`CutMix` (via `keras_cv.layers.MixUp` /
> `keras_cv.layers.CutMix`), and stronger color jitter. These regularize much harder and pay off
> once you have the epoch budget for them to not just add noise.


<a id="7"></a>
## 7. The Model Zoo -- 5 Architectures

We build and compare **5 models**:

| # | Model | Type | Loss | Purpose |
|---|---|---|---|---|
| 1 | **ResNet50** | Transfer learning (ImageNet) | Categorical Cross-Entropy | Strong, standard baseline |
| 2 | **DenseNet121** | Transfer learning (ImageNet) | Categorical Cross-Entropy | Dense connections -> efficient gradient flow |
| 3 | **Vision Transformer (Tiny/Fast)** | From scratch, patch-based transformer | Categorical Cross-Entropy | Modern attention-based architecture, no conv inductive bias |
| 4 | **`MARS_AI_Architecture`** | Our custom CNN, from scratch | Categorical Cross-Entropy | Lightweight, purpose-built for this task |
| 5 | **ResNet50 + Focal Loss** | Transfer learning (ImageNet) | **Custom Focal Loss** | Same backbone as Model 1, engineered for class imbalance |

All backbones are used with the classification "head" replaced and, for the fast demo,
**most of the backbone frozen** (only fine-tuning the head + last block) so training finishes
in minutes. See the tips for how to unfreeze more for a stronger final model.


### 7.1 Model 1 -- ResNet50 (Transfer Learning)

ResNet ("Residual Network") solves the vanishing-gradient problem in deep CNNs with **skip
connections**: `output = F(x) + x` instead of `output = F(x)`. This lets gradients flow directly
through the identity path during backprop, enabling much deeper networks to train successfully.

> Note: there is no official "ResNet121" variant; the standard sizes are **ResNet50/101/152**.
> We use **ResNet50** here as the fast baseline (swap in `ResNet101`/`ResNet152` for more capacity).


In [ ]:
def build_resnet50(num_classes=NUM_CLASSES, img_size=IMG_SIZE, unfreeze_last_n=10):
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(img_size, img_size, 3))
    base.trainable = True
    for layer in base.layers[:-unfreeze_last_n]:
        layer.trainable = False   # freeze most of the backbone -> fast training

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = data_augmentation(inputs)
    x = layers.Lambda(resnet_preprocess, name="resnet_preprocess")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="ResNet50_Baseline")

model_resnet = build_resnet50()
model_resnet.summary()


### 7.2 Model 2 -- DenseNet121 (Transfer Learning)

DenseNet connects **every layer to every other layer** in a feed-forward fashion within a
dense block (`x_l = H_l([x_0, x_1, ..., x_{l-1}])` -- concatenation, not addition). This
maximizes feature reuse and often gives strong accuracy with fewer parameters than ResNet.


In [ ]:
def build_densenet121(num_classes=NUM_CLASSES, img_size=IMG_SIZE, unfreeze_last_n=10):
    base = DenseNet121(weights="imagenet", include_top=False, input_shape=(img_size, img_size, 3))
    base.trainable = True
    for layer in base.layers[:-unfreeze_last_n]:
        layer.trainable = False

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = data_augmentation(inputs)
    x = layers.Lambda(densenet_preprocess, name="densenet_preprocess")(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="DenseNet121_Baseline")

model_densenet = build_densenet121()
model_densenet.summary()


### 7.3 Model 3 -- Vision Transformer (Fast/Tiny variant)

A full ViT-Base is overkill (and slow) for a 4-class, small-dataset demo. We implement a
**tiny ViT from scratch**: split the image into patches, linearly embed them, add positional
embeddings, and pass through a couple of Transformer encoder blocks (Multi-Head Self-Attention
+ MLP). This is fast enough to train from random init in a few minutes on a small dataset.

**Core idea:** an image of size `H x W` is split into `N = (H/P) x (W/P)` patches of size `P x P`,
each flattened and linearly projected to a `D`-dim embedding -- exactly like tokens in NLP
Transformers. Self-attention lets every patch attend to every other patch, giving ViT a global
receptive field from layer 1 (unlike CNNs, which build up receptive field gradually).


In [ ]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding="valid")

    def call(self, x):
        x = self.proj(x)                                   # (B, H/P, W/P, D)
        B = tf.shape(x)[0]
        D = x.shape[-1]
        x = tf.reshape(x, [B, -1, D])                       # (B, N, D)
        return x

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn  = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp   = keras.Sequential([
            layers.Dense(mlp_dim, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout),
        ])

    def call(self, x, training=False):
        x = x + self.attn(self.norm1(x), self.norm1(x), training=training)
        x = x + self.mlp(self.norm2(x), training=training)
        return x

def build_vit_tiny(num_classes=NUM_CLASSES, img_size=IMG_SIZE,
                    patch_size=16, embed_dim=64, num_heads=4, mlp_dim=128, num_blocks=4, dropout=0.1):
    num_patches = (img_size // patch_size) ** 2

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    x = PatchEmbedding(patch_size, embed_dim)(x)             # (B, N, D)

    cls_token = tf.Variable(tf.zeros((1, 1, embed_dim)), trainable=True, name="cls_token")
    pos_embed = tf.Variable(tf.random.normal((1, num_patches + 1, embed_dim), stddev=0.02), trainable=True, name="pos_embed")

    def add_cls_and_pos(x):
        B = tf.shape(x)[0]
        cls = tf.broadcast_to(cls_token, [B, 1, embed_dim])
        x = tf.concat([cls, x], axis=1)
        x = x + pos_embed
        return x

    x = layers.Lambda(add_cls_and_pos, name="cls_pos_embed")(x)
    x = layers.Dropout(dropout)(x)

    for i in range(num_blocks):
        x = TransformerBlock(embed_dim, num_heads, mlp_dim, dropout, name=f"transformer_block_{i}")(x)

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    cls_out = layers.Lambda(lambda t: t[:, 0], name="take_cls_token")(x)   # use CLS token for classification
    x = layers.Dense(64, activation="relu")(cls_out)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="ViT_Tiny_FromScratch")

model_vit = build_vit_tiny()
model_vit.summary()


> PRODUCTION TIP: ViTs are famously **data-hungry** (they lack CNNs' built-in
> translation/locality inductive bias, so they need more data or heavier augmentation to
> generalize). For production: (a) pretrain on ImageNet-21k and fine-tune (`keras_cv.models.ViTTiny16`
> or HuggingFace `transformers` `TFViTModel`), or (b) use much stronger augmentation (RandAugment,
> MixUp) and more epochs if training from scratch.


### 7.4 Model 4 -- `MARS_AI_Architecture` (Our Custom CNN)

A lightweight, purpose-built CNN designed for this specific 4-class problem: small enough to
train from scratch fast, with **Batch Normalization** for stable/faster convergence and
**Global Average Pooling** (instead of a big Flatten+Dense) to keep the parameter count low and
reduce overfitting risk on a small dataset.


In [ ]:
def MARS_AI_Architecture(num_classes=NUM_CLASSES, img_size=IMG_SIZE, dropout=0.3):
    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    # Block 1
    x = layers.Conv2D(32, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Block 4 (named so Grad-CAM can target it later)
    x = layers.Conv2D(128, 3, padding="same", activation=None, name="mars_last_conv")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(dropout / 2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="MARS_AI_Architecture")

model_mars = MARS_AI_Architecture()
model_mars.summary()


### 7.5 Model 5 -- ResNet50 + Custom Focal Loss (Class-Imbalance Specialist)

**Same exact architecture as Model 1**, but trained with a **custom Focal Loss** instead of
plain Categorical Cross-Entropy. This is a controlled experiment: any performance difference
between Model 1 and Model 5 is attributable to the **loss function**, not the architecture.
The full math is in Section 8 below.


In [ ]:
model_resnet_focal = build_resnet50()
model_resnet_focal._name = "ResNet50_FocalLoss"
model_resnet_focal.summary()


<a id="8"></a>
## 8. Deep Dive: Custom Loss Functions & the Math of Focal Loss

### 8.1 Why not just use Categorical Cross-Entropy everywhere?

Standard **Categorical Cross-Entropy (CCE)** for one sample is:

$$
{L}_{CE} = -\sum_{c=1}^{C} y_c \log(p_c)
$$

where $y_c \in \{0,1\}$ is the one-hot ground truth and $p_c$ is the predicted probability for
class $c$. Since $y$ is one-hot, this collapses to ${L}_{CE} = -\log(p_t)$, where $p_t$
is the model's predicted probability for the **true** class.

**The problem under class imbalance:** if class `back` has 3x more images than `left`, the
model can get a *low average loss* just by being great at `back` and mediocre at `left` -- the
majority class dominates the gradient signal during training. CCE treats every sample equally,
regardless of whether it's already easy (confidently correct) or hard (confused).

### 8.2 Focal Loss -- the fix

Focal Loss (Lin et al., *"Focal Loss for Dense Object Detection"*, 2017) adds a
**modulating factor** that down-weights *easy* (already well-classified) examples so training
focuses on *hard* / minority examples:

$$
{L}_{FL} = -\alpha_t (1 - p_t)^{\gamma} \log(p_t)
$$

Let's unpack every term:

| Symbol | Meaning | Intuition |
|---|---|---|
| $p_t$ | predicted probability for the **true** class | how confident & correct the model already is |
| $(1-p_t)^{\gamma}$ | **modulating factor** | if $p_t \to 1$ (easy, correct), this $\to 0$, loss suppressed. If $p_t \to 0$ (hard/wrong), this $\to 1$, loss stays large |
| $\gamma$ (gamma) | **focusing parameter**, typically 2.0 | higher gamma = more aggressive down-weighting of easy examples |
| $\alpha_t$ | **class-balance weight** for class $t$ | gives more overall weight to rare classes (like `left`), independent of how easy/hard the sample is |

**Worked numeric example** -- suppose $\gamma = 2$:
- **Easy, confident, correct sample:** $p_t = 0.9$ -> modulating factor $= (1-0.9)^2 = 0.01$ -> loss scaled down to **1%** of what plain CE would give it. The model stops spending gradient budget re-learning what it already knows.
- **Hard/misclassified sample:** $p_t = 0.2$ -> modulating factor $= (1-0.2)^2 = 0.64$ -> loss stays at **64%** of full strength. The model keeps getting a strong gradient push to fix this.

This is *exactly* what we want for imbalanced data: automatically **reweight training** toward
the minority-class / confusing examples (e.g. `left` vs `right`, which mostly differ by mirror
symmetry and are easy to confuse) instead of the easy majority class.

### 8.3 Multi-class implementation detail

For multi-class softmax outputs (our case, 4 classes) we compute Focal Loss **per class** and
sum, using one-hot targets:

$$
{L}_{FL} = -\sum_{c=1}^{C} \alpha_c \, y_c \, (1 - p_c)^{\gamma} \log(p_c)
$$

We set $\alpha_c$ using the **inverse class frequency** (the same `CLASS_WEIGHTS` computed in
Section 5.4 with `sklearn.utils.class_weight.compute_class_weight`), so rarer classes get a
bigger $\alpha_c$.


In [ ]:
def make_focal_loss(alpha, gamma=2.0):
    '''
    Categorical Focal Loss factory.

    Args:
        alpha: array-like of shape (num_classes,) -- per-class weight (alpha_t in the formula).
               Typically set from inverse class frequency (see CLASS_WEIGHTS above).
        gamma: focusing parameter (>= 0). gamma=0 reduces exactly to weighted CCE.

    Returns:
        A loss function usable directly in model.compile(loss=...).
    '''
    alpha = tf.constant(alpha, dtype=tf.float32)

    def focal_loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)   # numerical stability, avoid log(0)

        cross_entropy = -y_true * tf.math.log(y_pred)         # standard CE term, per class
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)  # prob of the TRUE class
        modulating_factor = tf.pow(1.0 - p_t, gamma)          # (1 - p_t) ^ gamma

        loss = alpha * modulating_factor * cross_entropy
        return tf.reduce_sum(loss, axis=-1)                   # sum over classes -> per-sample loss

    return focal_loss

# Build the alpha vector in class-index order from our computed class weights
alpha_vec = np.array([CLASS_WEIGHTS[i] for i in range(NUM_CLASSES)], dtype=np.float32)
alpha_vec = alpha_vec / alpha_vec.sum() * NUM_CLASSES   # normalize so avg weight ~ 1
print("Focal-loss alpha per class:", dict(zip(CLASS_NAMES, alpha_vec.round(3))))

focal_loss_fn = make_focal_loss(alpha=alpha_vec, gamma=2.0)

# ---- Sanity check: focal loss should be SMALLER than CCE for an easy/confident correct sample,
#      and roughly the SAME for a hard/wrong sample ----
y_true_demo = tf.constant([[0., 1., 0., 0.]])
y_pred_easy = tf.constant([[0.02, 0.94, 0.02, 0.02]])   # confident & correct
y_pred_hard = tf.constant([[0.30, 0.28, 0.22, 0.20]])   # barely-right / confused

cce = tf.keras.losses.CategoricalCrossentropy()
print("Easy sample -> CCE:", float(cce(y_true_demo, y_pred_easy)), "| Focal:", float(focal_loss_fn(y_true_demo, y_pred_easy)))
print("Hard sample -> CCE:", float(cce(y_true_demo, y_pred_hard)), "| Focal:", float(focal_loss_fn(y_true_demo, y_pred_hard)))


Notice how, for the **easy** sample, Focal Loss shrinks much more aggressively relative to
CCE than it does for the **hard** sample -- exactly the re-weighting behaviour derived above.

> 🔧 **PRODUCTION TIP:** Tune $\gamma$ (try 1.0-3.0) and $\alpha$ on a validation set. Also
> consider **Class-Balanced Loss** (Cui et al., 2019, using the "effective number of samples")
> as an alternative to plain inverse-frequency $\alpha$ when class imbalance is severe (>10x).


<a id="9"></a>
## 9. Training Loop -- All 5 Models

We now build the datasets (one per preprocessing scheme) and train all 5 models with a shared,
lightweight training recipe: `Adam` optimizer, `EarlyStopping` on val loss (so a model that
converges fast doesn't waste your class time), and a `ModelCheckpoint` keeping the best weights.

> 🔧 **PRODUCTION TIP:** for a long-training-budget run, replace `EPOCHS_DEMO=4` with
> `EPOCHS=100`, increase `patience` in `EarlyStopping` to ~15-20, add
> `ReduceLROnPlateau`, unfreeze the **entire** backbone after a few warmup epochs
> (progressive/fine-tuning "discriminative learning rates" -- lower LR for early backbone
> layers, higher LR for the head), and train on the full-resolution `IMG_SIZE=224`. That
> combination is what reliably gets you a robust, well-generalizing production model --
> it just costs hours instead of minutes.


In [ ]:
# ---- Build one-hot label datasets (needed for categorical losses / focal loss) ----
def make_dataset_onehot(df, preprocess_fn=None, shuffle=False, augment=False):
    ds = make_dataset(df, shuffle=shuffle, augment=augment, preprocess_fn=None)  # preprocessing happens INSIDE each model now
    def _to_onehot(img, label):
        return img, tf.one_hot(label, NUM_CLASSES)
    return ds.map(_to_onehot, num_parallel_calls=AUTOTUNE)

# NOTE: preprocessing (rescale / imagenet-style) is baked into each model's graph (Lambda /
# Rescaling layers) via the model-builder functions above, so the datasets themselves just
# need raw resized images in [0, 255].
train_ds = make_dataset_onehot(train_df, shuffle=True)
val_ds   = make_dataset_onehot(val_df)
test_ds  = make_dataset_onehot(test_df)

print("Datasets ready.")


In [ ]:
def compile_and_train(model, loss, train_ds, val_ds, epochs=EPOCHS_DEMO, lr=1e-3, name=None):
    name = name or model.name
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=loss,
        metrics=["accuracy"]
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(f"{name}_best.keras", monitor="val_loss", save_best_only=True),
    ]

    t0 = time.time()
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=epochs, callbacks=callbacks, verbose=1
    )
    elapsed = time.time() - t0
    print(f"[{name}] training finished in {elapsed:.1f}s ({elapsed/60:.2f} min)")
    return history, elapsed

histories = {}
timings = {}


In [ ]:
# ---- Model 1: ResNet50 (Cross-Entropy) ----
print("="*70, "\nTraining Model 1: ResNet50 (Cross-Entropy)\n", "="*70)
hist, t = compile_and_train(model_resnet, "categorical_crossentropy", train_ds, val_ds, name="ResNet50")
histories["ResNet50"] = hist
timings["ResNet50"] = t


In [ ]:
# ---- Model 2: DenseNet121 (Cross-Entropy) ----
print("="*70, "\nTraining Model 2: DenseNet121 (Cross-Entropy)\n", "="*70)
hist, t = compile_and_train(model_densenet, "categorical_crossentropy", train_ds, val_ds, name="DenseNet121")
histories["DenseNet121"] = hist
timings["DenseNet121"] = t


In [ ]:
# ---- Model 3: Vision Transformer (Cross-Entropy, from scratch, slightly more epochs since it has no pretrained prior) ----
print("="*70, "\nTraining Model 3: ViT-Tiny (Cross-Entropy)\n", "="*70)
hist, t = compile_and_train(model_vit, "categorical_crossentropy", train_ds, val_ds,
                             epochs=EPOCHS_DEMO + 2, lr=1e-3, name="ViT_Tiny")
histories["ViT_Tiny"] = hist
timings["ViT_Tiny"] = t


In [ ]:
# ---- Model 4: MARS_AI_Architecture (Cross-Entropy, from scratch) ----
print("="*70, "\nTraining Model 4: MARS_AI_Architecture (Cross-Entropy)\n", "="*70)
hist, t = compile_and_train(model_mars, "categorical_crossentropy", train_ds, val_ds,
                             epochs=EPOCHS_DEMO + 2, name="MARS_AI_Architecture")
histories["MARS_AI_Architecture"] = hist
timings["MARS_AI_Architecture"] = t


In [ ]:
# ---- Model 5: ResNet50 + Custom Focal Loss ----
print("="*70, "\nTraining Model 5: ResNet50 + Focal Loss\n", "="*70)
hist, t = compile_and_train(model_resnet_focal, focal_loss_fn, train_ds, val_ds, name="ResNet50_FocalLoss")
histories["ResNet50_FocalLoss"] = hist
timings["ResNet50_FocalLoss"] = t


In [ ]:
# ---- Training curves for all 5 models ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, hist in histories.items():
    axes[0].plot(hist.history["loss"], label=f"{name} (train)")
    axes[0].plot(hist.history["val_loss"], "--", label=f"{name} (val)")
    axes[1].plot(hist.history["accuracy"], label=f"{name} (train)")
    axes[1].plot(hist.history["val_accuracy"], "--", label=f"{name} (val)")

axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(fontsize=7)
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

print("\nTraining time per model:")
for name, t in timings.items():
    print(f"  {name:22s}: {t:6.1f}s ({t/60:.2f} min)")
print(f"\nTOTAL: {sum(timings.values()):.1f}s ({sum(timings.values())/60:.2f} min)")


<a id="10"></a>
## 10. 📊 Model Comparison -- Accuracy, Precision, Recall, F1, AUC/ROC

**Why we track more than accuracy:** under class imbalance, a model can score high accuracy by
just being good at the majority class. So we always report the full picture:

- **Precision** = $\frac{TP}{TP+FP}$ -- "of everything I called `front`, how much really was `front`?"
- **Recall** = $\frac{TP}{TP+FN}$ -- "of all the real `front` images, how many did I catch?"
- **F1** = harmonic mean of precision & recall -- balances both, good single-number summary per class
- **AUC / ROC** -- how well the model *ranks* the correct class above incorrect ones across all thresholds, one-vs-rest for multi-class

We use **macro-averaging** (unweighted mean across classes) so every class -- majority or
minority -- counts equally in the summary metric, which is exactly what matters under imbalance.


In [ ]:
all_models = {
    "ResNet50": model_resnet,
    "DenseNet121": model_densenet,
    "ViT_Tiny": model_vit,
    "MARS_AI_Architecture": model_mars,
    "ResNet50_FocalLoss": model_resnet_focal,
}

y_test_true = test_df["label_idx"].values
y_test_true_onehot = tf.one_hot(y_test_true, NUM_CLASSES).numpy()

results = {}
predictions = {}

for name, model in all_models.items():
    y_pred_proba = model.predict(test_ds, verbose=0)
    y_pred_idx = np.argmax(y_pred_proba, axis=1)
    predictions[name] = {"proba": y_pred_proba, "pred_idx": y_pred_idx}

    acc  = accuracy_score(y_test_true, y_pred_idx)
    prec = precision_score(y_test_true, y_pred_idx, average="macro", zero_division=0)
    rec  = recall_score(y_test_true, y_pred_idx, average="macro", zero_division=0)
    f1   = f1_score(y_test_true, y_pred_idx, average="macro", zero_division=0)
    try:
        auc_score = roc_auc_score(y_test_true_onehot, y_pred_proba, average="macro", multi_class="ovr")
    except ValueError:
        auc_score = np.nan

    results[name] = {
        "Accuracy": acc, "Precision (macro)": prec, "Recall (macro)": rec,
        "F1 (macro)": f1, "AUC (macro, OVR)": auc_score,
        "Train time (s)": timings[name],
    }

results_df = pd.DataFrame(results).T.sort_values("F1 (macro)", ascending=False)
results_df_display = results_df.copy()
for col in ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)", "AUC (macro, OVR)"]:
    results_df_display[col] = results_df_display[col].map(lambda x: f"{x:.4f}")
display(results_df_display)

BEST_MODEL_NAME = results_df.index[0]
print(f"\n🏆 Best model by macro F1: {BEST_MODEL_NAME}")


In [ ]:
# ---- Bar chart comparison across all metrics ----
metrics_to_plot = ["Accuracy", "Precision (macro)", "Recall (macro)", "F1 (macro)", "AUC (macro, OVR)"]
plot_df = results_df[metrics_to_plot].astype(float)

ax = plot_df.plot(kind="bar", figsize=(13, 6), width=0.8)
plt.title("Model comparison across metrics (test set)")
plt.ylabel("score")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1.05)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ---- ROC curves (one-vs-rest) for the BEST model ----
best_proba = predictions[BEST_MODEL_NAME]["proba"]

plt.figure(figsize=(7, 6))
for i, c in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_test_true_onehot[:, i], best_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{c} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC curves (one-vs-rest) -- {BEST_MODEL_NAME}")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---- Full per-class classification report for the best model ----
print(f"Classification report -- {BEST_MODEL_NAME}\n")
print(classification_report(y_test_true, predictions[BEST_MODEL_NAME]["pred_idx"], target_names=CLASS_NAMES, digits=4))


<a id="11"></a>
## 11. 🔍 Error Analysis -- Confusion Matrix & Misclassified Samples

Metrics tell you *how much* the model is wrong. Error analysis tells you *why*. We look at:
1. The confusion matrix -- which classes get mixed up with which?
2. Actual misclassified images -- e.g. does a `back` photo get called `front` because the bike
   is symmetric-looking from behind, or because of bad lighting?


In [ ]:
cm = confusion_matrix(y_test_true, predictions[BEST_MODEL_NAME]["pred_idx"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix -- {BEST_MODEL_NAME}")
plt.tight_layout()
plt.show()

# Which pair is most confused (excluding the diagonal)?
cm_noeye = cm.copy().astype(float)
np.fill_diagonal(cm_noeye, 0)
i, j = np.unravel_index(np.argmax(cm_noeye), cm_noeye.shape)
if cm_noeye[i, j] > 0:
    print(f"Most common confusion: true='{CLASS_NAMES[i]}' predicted as '{CLASS_NAMES[j]}' ({int(cm_noeye[i,j])} times)")


In [ ]:
# ---- Visualize actual misclassified test images ----
test_df_reset = test_df.reset_index(drop=True)
pred_idx = predictions[BEST_MODEL_NAME]["pred_idx"]
pred_proba = predictions[BEST_MODEL_NAME]["proba"]

wrong_mask = pred_idx != test_df_reset["label_idx"].values
wrong_indices = np.where(wrong_mask)[0]
print(f"{len(wrong_indices)} / {len(test_df_reset)} test images misclassified by {BEST_MODEL_NAME}")

n_show = min(8, len(wrong_indices))
if n_show > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    for ax, idx in zip(axes, wrong_indices[:n_show]):
        row = test_df_reset.iloc[idx]
        img = mpimg.imread(row["filepath"])
        true_label = row["label"]
        pred_label = CLASS_NAMES[pred_idx[idx]]
        conf = pred_proba[idx][pred_idx[idx]]
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.2f})", color="red", fontsize=10)
    for ax in axes[n_show:]:
        ax.axis("off")
    plt.suptitle("Misclassified examples -- error analysis", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No misclassifications on the test set -- try a smaller/harder test split to see error patterns, or the model genuinely nailed it!")


**How to read this:** typical confusions in a viewpoint-classification task are between
visually-similar viewpoints -- e.g. `left` vs `right` (mirror-symmetric silhouettes, especially
if the motorcycle model has few asymmetric visual cues), or `front` vs `back` when the bike is
heavily silhouetted / backlit and fine details (headlight vs taillight) aren't visible. If you
see a systematic pattern like this, it tells you exactly what kind of extra training data or
augmentation would help most (e.g. more `left`/`right` examples with clearer asymmetric cues).


<a id="12"></a>
## 12. 🔥 Grad-CAM -- Visual Explainability

**Grad-CAM** (Gradient-weighted Class Activation Mapping, Selvaraju et al. 2017) answers:
*"which pixels made the model predict this class?"* It works by:

1. Running a forward pass and getting the gradient of the predicted class score w.r.t. the
   **last convolutional layer's** feature maps.
2. Global-average-pooling those gradients to get an "importance weight" per feature-map channel.
3. Computing a weighted sum of the feature maps, then ReLU (we only care about positive
   influence on the class), giving a coarse heatmap of "where the model looked."

$$
L^{c}_{Grad-CAM} = \text{ReLU}\left(\sum_k \alpha_k^c \, A^k\right), \quad
\alpha_k^c = \frac{1}{Z}\sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}}
$$

This only works cleanly on **convolutional** architectures (ResNet, DenseNet, our custom CNN) --
ViT uses attention, not convolutional feature maps, so its explainability method is
**attention-map visualization** instead (noted below, not implemented here for time).


In [ ]:
def get_last_conv_layer_name(model):
    '''Find the last Conv2D (or nested backbone's last conv) layer name for Grad-CAM.'''
    for layer in reversed(model.layers):
        # If it's a nested functional model (e.g. ResNet50 backbone), search inside it too
        if isinstance(layer, keras.Model):
            for inner in reversed(layer.layers):
                if isinstance(inner, layers.Conv2D):
                    return layer.name, inner.name
        if isinstance(layer, layers.Conv2D):
            return None, layer.name
    raise ValueError("No Conv2D layer found for Grad-CAM.")

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, nested_model_name=None, pred_index=None):
    '''
    NOTE on Keras 3: you canNOT build a single keras.Model that reaches directly into a nested
    sub-model's internal tensors (e.g. model.get_layer('resnet50').get_layer('conv5_...').output)
    -- Keras 3's functional-graph tracking treats a nested Functional model as one opaque layer,
    so wiring its INNER tensor into an OUTER Model's outputs list raises a KeyError at call time.

    Fix: keep the two graphs separate and glue them together at runtime, inside a single
    GradientTape, using plain layer calls. Gradients still flow correctly end-to-end because
    it's all one continuous eager computation under the tape -- we just avoid asking Keras to
    build one static graph object that spans both scopes.
    '''
    if nested_model_name is not None:
        nested = model.get_layer(nested_model_name)
        conv_layer_output = nested.get_layer(last_conv_layer_name).output
        # This sub-model is valid: both tensors live INSIDE nested's own graph.
        nested_grad_model = keras.Model(nested.input, [conv_layer_output, nested.output])

        outer_layer_names = [l.name for l in model.layers]
        nested_idx = outer_layer_names.index(nested_model_name)
        pre_layers  = model.layers[1:nested_idx]     # layers between Input and the backbone (e.g. augmentation, preprocess Lambda)
        post_layers = model.layers[nested_idx + 1:]   # layers after the backbone (GAP, dropout, dense...)

        with tf.GradientTape() as tape:
            x = img_array
            for layer in pre_layers:
                x = layer(x, training=False)
            conv_out, nested_out = nested_grad_model(x, training=False)
            y = nested_out
            for layer in post_layers:
                y = layer(y, training=False)
            preds = y
            if pred_index is None:
                pred_index = tf.argmax(preds[0])
            class_channel = preds[:, pred_index]
        grads = tape.gradient(class_channel, conv_out)
    else:
        grad_model = keras.Model(inputs=model.input, outputs=[model.get_layer(last_conv_layer_name).output, model.output])
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(img_array)
            if pred_index is None:
                pred_index = tf.argmax(preds[0])
            class_channel = preds[:, pred_index]
        grads = tape.gradient(class_channel, conv_out)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def overlay_gradcam(img_uint8, heatmap, alpha=0.4):
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (img_uint8.shape[0], img_uint8.shape[1])).numpy().squeeze()
    heatmap_color = plt.cm.jet(heatmap_resized)[..., :3] * 255
    overlay = (heatmap_color * alpha + img_uint8 * (1 - alpha)).astype("uint8")
    return overlay


In [ ]:
# ---- Run Grad-CAM on a few TEST images using our best CONV-based model ----
# If the best model is ViT (no conv layers), fall back to MARS_AI_Architecture for the Grad-CAM demo.
gradcam_model_name = BEST_MODEL_NAME if BEST_MODEL_NAME != "ViT_Tiny" else "MARS_AI_Architecture"
gradcam_model = all_models[gradcam_model_name]
print("Running Grad-CAM using model:", gradcam_model_name)

nested_name, conv_name = get_last_conv_layer_name(gradcam_model)
print("Nested backbone:", nested_name, "| Last conv layer:", conv_name)

sample_rows = test_df_reset.sample(n=min(4, len(test_df_reset)), random_state=SEED)

fig, axes = plt.subplots(2, len(sample_rows), figsize=(4*len(sample_rows), 8))
for col, (_, row) in enumerate(sample_rows.iterrows()):
    img_raw = tf.io.decode_jpeg(tf.io.read_file(row["filepath"]), channels=3)
    img_resized = tf.image.resize(img_raw, [IMG_SIZE, IMG_SIZE])
    img_batch = tf.expand_dims(img_resized, 0)

    heatmap, pred_idx_gc = make_gradcam_heatmap(img_batch, gradcam_model, conv_name, nested_model_name=nested_name)
    img_uint8 = img_resized.numpy().astype("uint8")
    overlay = overlay_gradcam(img_uint8, heatmap)

    axes[0, col].imshow(img_uint8); axes[0, col].axis("off")
    axes[0, col].set_title(f"True: {row['label']}", fontsize=10)
    axes[1, col].imshow(overlay); axes[1, col].axis("off")
    axes[1, col].set_title(f"Grad-CAM -> Pred: {CLASS_NAMES[pred_idx_gc]}", fontsize=10)

plt.suptitle(f"Grad-CAM explainability -- {gradcam_model_name}", fontsize=14)
plt.tight_layout()
plt.show()


**How to read this:** a well-trained model's heatmap should light up on the **motorcycle
itself** -- headlight/handlebars for `front`, taillight/exhaust for `back`, wheel silhouette
profile for `left`/`right`. If the heatmap instead lights up on the **background** (a wall, a
watermark, a person standing nearby), that's a red flag the model may be learning spurious
shortcuts rather than the actual motorcycle shape -- a classic thing Grad-CAM helps you catch
that a plain accuracy number never would.

> 🔧 **PRODUCTION TIP:** for ViT explainability, use **attention rollout** or the
> `[CLS]` token's attention weights over patches (visualized as a patch-grid heatmap) instead of
> Grad-CAM -- conceptually similar goal ("where did the model look"), different mechanism.


<a id="13"></a>
## 13. 🌍 Mock Real-World Inference

The ultimate sanity check: pull a motorcycle photo straight from the internet (not from our
dataset at all) and see what the best model predicts.


In [ ]:
def predict_from_url(url, model, img_size=IMG_SIZE, class_names=CLASS_NAMES, show=True):
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    img = Image.open(io.BytesIO(resp.content)).convert("RGB")

    img_resized = img.resize((img_size, img_size))
    img_array = np.array(img_resized, dtype=np.float32)
    img_batch = np.expand_dims(img_array, axis=0)

    proba = model.predict(img_batch, verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    pred_label = class_names[pred_idx]

    if show:
        plt.figure(figsize=(5, 5))
        plt.imshow(img)
        plt.axis("off")
        conf_str = ", ".join([f"{c}:{p:.2f}" for c, p in zip(class_names, proba)])
        plt.title(f"Predicted: {pred_label.upper()} (conf={proba[pred_idx]:.2f})\n{conf_str}", fontsize=10)
        plt.tight_layout()
        plt.show()

    return pred_label, proba

demo_url = "https://github.com/kaopanboonyuen/MARS-LEARN/raw/main/sample_img/Kawasaki_Ninja_400_KRT_Edition.jpg"
best_model = all_models[BEST_MODEL_NAME]
pred_label, proba = predict_from_url(demo_url, best_model)

print(f"\nModel used: {BEST_MODEL_NAME}")
print(f"Predicted viewpoint: {pred_label}")
print("Per-class probabilities:", dict(zip(CLASS_NAMES, proba.round(3))))
print("\n(Sanity check: the filename literally says 'right_side' -- a good model should predict 'right'.)")


> 🔧 **PRODUCTION TIP:** for real deployment, wrap this in **Test-Time Augmentation (TTA)** --
> run inference on a few augmented copies (small rotations/crops) of the input and average the
> softmax outputs. This typically gives a small but reliable accuracy boost at inference time,
> at the cost of a few extra forward passes.


<a id="14"></a>
## 14. 🎓 Key Takeaways & Best-Practice Cheat Sheet

1. **Always look at your data before modeling.** The class-imbalance we found in Section 4
   directly justified building Model 5 with Focal Loss -- EDA drives modeling decisions, not
   the other way around.
2. **Think about label-safety before choosing augmentations.** A horizontal flip is "standard"
   almost everywhere -- except when your classes are literally defined by left/right symmetry.
3. **No single architecture wins by default.** Transfer-learning CNNs (ResNet/DenseNet) usually
   win on small datasets thanks to ImageNet-pretrained features; ViT needs more data/epochs to
   shine; a small custom CNN can be a surprisingly strong, cheap baseline.
4. **Loss function choice is a real lever, not an afterthought.** Comparing Model 1 vs Model 5
   (identical architecture, different loss) isolates *exactly* how much Focal Loss helped with
   the imbalance -- a clean, convincing experiment to show a team.
5. **Accuracy alone lies under imbalance.** Always report macro Precision/Recall/F1/AUC so
   minority classes aren't hidden by majority-class performance.
6. **Error analysis and Grad-CAM catch things metrics can't.** A high-accuracy model that's
   secretly keying off the background/watermark is a ticking time bomb in production --
   explainability tooling is how you catch it *before* deployment.
7. **A fast demo notebook and a production notebook are different artifacts.** This notebook
   is tuned for a **5-15 min class demo**; every 🔧 tip marks exactly what to scale up
   (epochs, image size, unfreezing, augmentation strength, dataset size) for a robust,
   production-grade model.


<a id="15"></a>
## 15. 🚀 Appendix: Production-Robustness Tricks

A grab-bag of techniques to apply when you have a real training budget (hours/days instead of
minutes) and want a genuinely robust, well-generalizing model:

### 15.1 Add a "negative" / background class
Real-world inference will eventually see images that are **not** a clean motorcycle photo at
all (a car, a person, a blank wall, a screenshot). If your model only ever saw 4 motorcycle
viewpoints, it will confidently mis-classify these as one of the 4 classes -- softmax always
sums to 1, it *has* to pick something. Fix: add a 5th `negative`/`unknown` class populated with:
- A random sample of **ImageNet** validation images (`tf.keras.utils.image_dataset_from_directory`
  on a downloaded ImageNet subset, or `tensorflow_datasets.load('imagenet_v2')`)
- Random crops from **COCO** or **Open Images** that don't contain vehicles
- Optionally, hard negatives: cars, bicycles, scooters (visually close but wrong category)

```python
# Sketch: merging in a negative class
negative_ds = tf.keras.utils.image_dataset_from_directory(
    "path/to/imagenet_subset", image_size=(IMG_SIZE, IMG_SIZE), label_mode=None
)
negative_ds = negative_ds.map(lambda x: (x, tf.one_hot(NUM_CLASSES, NUM_CLASSES + 1)))  # new 5th class index
# Then concatenate with the motorcycle dataset and rebuild the final Dense(num_classes+1) layer.
```

### 15.2 Longer, smarter training schedules
- Two-phase fine-tuning: **Phase 1** -- freeze the whole backbone, train only the new head for
  a few epochs (what we did here). **Phase 2** -- unfreeze the entire backbone, drop the
  learning rate by 10x, and continue training for many more epochs. This lets the pretrained
  features adapt to motorcycle-specific textures without catastrophically forgetting.
- `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5)` and/or cosine-decay LR schedules.
- `EarlyStopping(patience=15-20)` with `restore_best_weights=True` so long runs don't overfit.

### 15.3 Stronger augmentation for small datasets
- `RandAugment`, `MixUp`, `CutMix` (all available via `keras_cv.layers`) -- especially important
  for the ViT, which has no built-in locality bias and needs the extra regularization more than
  the CNNs do.
- Domain-realistic augmentations: simulate different backgrounds (copy-paste the motorcycle
  onto random backgrounds), simulate different lighting/weather, simulate motion blur.

### 15.4 More rigorous evaluation
- **k-fold cross-validation** instead of a single train/val/test split, if the dataset is small
  (more reliable metric estimates, less sensitive to a "lucky" split).
- **Test-Time Augmentation (TTA)** at inference (Section 13 tip).
- Track a **held-out "hard" test set** curated specifically from ambiguous/ difficult images
  (poor lighting, partial occlusion, unusual angles) to stress-test beyond the easy majority.

### 15.5 Model efficiency for deployment
- Once you've picked a winner, apply **post-training quantization** (`tf.lite`) or
  **pruning** (`tensorflow_model_optimization`) if this needs to run on-device / at low latency.
- Consider **knowledge distillation**: train a small student model (like `MARS_AI_Architecture`)
  to mimic the best big model's soft predictions -- often recovers most of the accuracy at a
  fraction of the inference cost.

### 15.6 Monitoring after deployment
- Log prediction confidence distributions in production; a sudden shift usually signals
  **data drift** (new camera, new motorcycle models, new backgrounds) before accuracy visibly drops.
- Periodically spot-check Grad-CAM on live traffic samples, not just the original test set.

---

### 📌 Summary of every knob mentioned in this notebook

| Knob | Demo value (fast) | Production value (robust) |
|---|---|---|
| `IMG_SIZE` | 128 | 224 (matches ImageNet pretraining) |
| `EPOCHS_DEMO` | 4-6 | 50-100+ with EarlyStopping |
| Backbone unfreezing | last 10 layers only | full backbone, phase 2 fine-tune, low LR |
| Augmentation | mild rotate/zoom/brightness | + RandAugment/MixUp/CutMix |
| Dataset | as-is | + negative class (ImageNet/COCO), more raw images |
| Evaluation | single split | k-fold CV + TTA + hard-case test set |
| Loss | CCE vs Focal (compared here) | tuned $\gamma$/$\alpha$, or Class-Balanced Loss |

---

*Notebook by Kao Panboonyuen -- built as a teaching resource for the MARS-AI team.* 🏍️
